# Практическая работа №3. Распределённая модель обработки данных

**Дисциплина:** Технологии и инструментарий анализа больших данных  
**Тема:** Введение в разработку аналитического сценария на PySpark с внешней базой данных

| | |
|---|---|
| **Выполнил:** | Тагин Никита |
| **Группа:** | ИКБО-61-23 |
| **Дата выполнения:** | 28.10.2026 |
| **Проверил:** | ст. преподаватель Приходько Н.А. |

**Цель работы** — освоить базовый цикл распределённой обработки табличных данных средствами Apache Spark и PySpark:
запустить локальное Spark-приложение, подготовить исходные таблицы из базы SQLite, прочитать их в DataFrame, исследовать
и задать схему, выполнить преобразования, обработать пропуски и типы, сформировать агрегаты, соединить связанные таблицы,
повторить аналитическую задачу через Spark SQL и сохранить итоговый набор данных.

Работа состоит из двух частей:
1. **Часть I** — учебные блоки на базе `ecommerce.sqlite` (интернет-магазин: заказы, позиции, товары, клиенты, платежи).
2. **Часть II** — самостоятельный распределённый анализ собственного набора «Транзакции КОМУС» (продажи + справочники товаров и клиентов).

В конце приведены ответы на контрольные вопросы.

# Часть I. Учебные блоки на базе ecommerce.sqlite

## 3. Рабочее окружение и подготовка источника
### Практический блок 3.1. Подготовка окружения

Работа выполнялась на macOS (Apple Silicon). Окружение создано в каталоге `.venv` рядом с блокнотом:

```bash
python3.13 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install pyspark "pandas<3" pyarrow jupyter "nbconvert[webpdf]"
python -m playwright install chromium   # движок для экспорта WebPDF
java -version
python -c "import pyspark; print(pyspark.__version__)"
jupyter notebook
```

На компьютере установлено несколько JDK (8, 21, 23). Методичка рекомендует JDK 17 или 21, поэтому в первой ячейке
`JAVA_HOME` явно указывает на JDK 21 — это делается **до** создания SparkSession, т.к. именно в этот момент PySpark запускает JVM.
`pandas` закреплён на ветке 2.x: PySpark 4.2 выдаёт предупреждение о неполной поддержке pandas 3.

Блокнот выполнен в веб-интерфейсе Jupyter Notebook (браузер Chrome, сервер `jupyter notebook` из `.venv`) командой
*Kernel → Restart Kernel and Run All Cells*; отчёт выгружен оттуда же через *File → Save and Export Notebook As → WebPDF*
(для этого дополнительно установлены `nbconvert[webpdf]` и браузер Chromium для Playwright). Сервер запущен с
`JUPYTER_PATH=jupyter_templates`: в этом каталоге лежит шаблон WebPDF, который наследует стандартный `lab` и добавляет
CSS — перенос длинных строк кода и уменьшенный шрифт вывода, чтобы код и таблицы Spark не обрезались по краю страницы.

In [1]:
import os, sys, platform, subprocess
from pathlib import Path

# JDK 21 (рекомендуемая методичкой LTS-версия); если каталога нет — используется системная Java
JDK21 = "/Users/nikita/Library/Java/JavaVirtualMachines/jbr-21.0.6/Contents/Home"
if Path(JDK21).is_dir():
    os.environ["JAVA_HOME"] = JDK21
    os.environ["PATH"] = f"{JDK21}/bin:" + os.environ["PATH"]
# Python-воркеры Spark должны использовать тот же интерпретатор, что и ядро Jupyter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# адрес драйвера и уровень журнала JVM: без этого в вывод ячеек попадают служебные WARN-сообщения при старте JVM
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
SPARK_CONF = Path("spark_conf")
SPARK_CONF.mkdir(exist_ok=True)
(SPARK_CONF / "log4j2.properties").write_text(
    "rootLogger.level = error\n"
    "rootLogger.appenderRef.stdout.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n\n"
)
os.environ["SPARK_CONF_DIR"] = str(SPARK_CONF.resolve())

import contextlib

JVM_LOG = Path("spark_logs/jvm_stderr.log")

@contextlib.contextmanager
def jvm_stderr_to_file(path=JVM_LOG):
    # JVM наследует stderr процесса Python в момент запуска; на это время stderr направляется в файл,
    # чтобы служебные строки JVM ("Using incubator modules", "Setting default log level") не попадали в ячейки.
    # Ошибки Spark при этом не теряются: они приходят в Python как исключения Py4J/PySpark.
    path.parent.mkdir(exist_ok=True)
    sys.stderr.flush()
    saved = os.dup(2)
    with open(path, "ab") as f:
        os.dup2(f.fileno(), 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)

import pyspark, pandas as pd

java_ver = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print("OS:     ", platform.platform())
print("Python: ", platform.python_version())
print("Java:   ", java_ver)
print("PySpark:", pyspark.__version__)
print("pandas: ", pd.__version__)

OS:      macOS-27.0.1-arm64-arm-64bit-Mach-O
Python:  3.13.2
Java:    openjdk version "21.0.6" 2025-01-21
PySpark: 4.2.1
pandas:  2.3.3


### Внешняя база `ecommerce.sqlite`

Источник данных — выданная SQLite-база `ecommerce.sqlite`. Её схема соответствует DDL из `ecommerce.sql`
(практическая работа №2): те же 11 пользовательских таблиц, первичные и внешние ключи, индексы. Кроме них в базе есть
служебные таблицы SQLite (`sqlite_sequence`, `sqlite_stat1`, `sqlite_stat4`). Перед staging база открывается только
на чтение (`mode=ro`), выводятся число строк в таблицах, число индексов и DDL центральной таблицы фактов `order_items`.

In [2]:
import zipfile, sqlite3

DB_PATH = Path("ecommerce.sqlite")
print("Размер базы:", round(DB_PATH.stat().st_size / 2**20, 1), "MB")

with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"]
    for t in tables:
        n = conn.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        kind = "служебная" if t.startswith("sqlite_") else ""
        print(f"{t:36s} {n:>9}  {kind}")
    n_idx = conn.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='index'").fetchone()[0]
    print("\nИндексов:", n_idx)
    print(conn.execute("SELECT sql FROM sqlite_master WHERE name='order_items'").fetchone()[0])

Размер базы: 200.9 MB
customers                                99441  
geolocation                            1000163  
leads_closed                               842  
leads_qualified                           8000  
order_items                             112650  
order_payments                          103886  
order_reviews                            97621  
orders                                   99441  
product_category_name_translation           71  
products                                 32951  
sellers                                   3095  
sqlite_sequence                              1  служебная
sqlite_stat1                                29  служебная
sqlite_stat4                               659  служебная

Индексов: 29
CREATE TABLE "order_items" (
  order_id TEXT NOT NULL,
  order_item_id INTEGER NOT NULL,
  product_id TEXT,
  seller_id TEXT,
  shipping_limit_date TEXT,
  price NUMERIC,
  freight_value NUMERIC,
  PRIMARY KEY (order_id, order_item_id),
  FOREIGN KEY 

### Практический блок 3.2. Извлечение таблиц SQLite в staging-каталог

Spark не читает файл SQLite как встроенный формат, поэтому используется воспроизводимый staging-этап:
`sqlite3` открывает базу, `pandas.read_sql_query` выполняет `SELECT *`, `to_csv(index=False)` пишет промежуточный CSV
в каталог `spark_input`. Таблица `geolocation` (~1 млн строк) в основной последовательности не извлекается.

In [3]:
STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

TABLES = [
    "orders",
    "order_items",
    "products",
    "customers",
    "order_payments",
    "order_reviews",
    "product_category_name_translation",
]

with sqlite3.connect(DB_PATH) as conn:
    for table in TABLES:
        part = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
        part.to_csv(STAGING / f"{table}.csv", index=False, encoding="utf-8")
        print(f"{table:36s} {len(part):>8} rows")

orders                                  99441 rows
order_items                            112650 rows
products                                32951 rows
customers                               99441 rows
order_payments                         103886 rows
order_reviews                           97621 rows
product_category_name_translation          71 rows


**Вывод.** В выданной базе 11 пользовательских таблиц и 3 служебные таблицы SQLite, 29 индексов; схема совпадает с `ecommerce.sql`. Staging перенёс 7 нужных таблиц в CSV без изменения структуры, число строк совпадает с методичкой. Самая крупная из извлечённых таблиц — `order_items` (112 650 строк). `geolocation` (1 000 163 строки) в staging сознательно не попала.

## 4. Запуск локального Spark
### Практический блок 4.1. Создание SparkSession

`master("local[*]")` разрешает Spark использовать все логические ядра компьютера, `spark.sql.shuffle.partitions=8`
уменьшает число мелких задач после shuffle. Дополнительно отключён Spark UI (`spark.ui.enabled=false`): путь к проекту
содержит кириллицу, и встроенный веб-сервер UI не может открыть свои статические ресурсы (на вычисления это не влияет,
но засоряет вывод предупреждениями). По той же причине выключен консольный индикатор стадий `showConsoleProgress`,
а в первой ячейке задан файл `spark_conf/log4j2.properties` с уровнем журнала JVM `error`. Служебные строки, которые
JVM печатает при запуске (`WARNING: Using incubator modules: jdk.incubator.vector` — Spark 4 подключает векторный модуль JDK;
`Setting default log level to "WARN"`), не являются ошибками; чтобы они не выглядели в отчёте как сбой, сессия создаётся
внутри `jvm_stderr_to_file()`, и этот вывод сохраняется в `spark_logs/jvm_stderr.log`.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

with jvm_stderr_to_file():
    spark = (
        SparkSession.builder
        .appName("Practice3_Ecommerce")
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)
print("defaultParallelism:", spark.sparkContext.defaultParallelism)

Master: local[*]
App: Practice3_Ecommerce
Spark: 4.2.1
defaultParallelism: 8


**Вывод.** Сессия запущена в режиме `local[*]` на одном компьютере: `defaultParallelism = 8`, т.е. Spark может выполнять 8 задач одновременно — по числу логических ядер. Модель Driver/Executor при этом эмулируется внутри одной JVM.

## 5. Чтение данных и работа со схемой
### Практический блок 5.1. Чтение orders без автоматического вывода типов

In [5]:
orders_raw = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "orders.csv"))
)

orders_raw.printSchema()
print("Строк:", orders_raw.count())
print("Разделов (partitions):", orders_raw.rdd.getNumPartitions())

orders_raw.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
).show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

Строк: 99441
Разделов (partitions): 5
+--------------------------------+--------------------------------+------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|
+--------------------------------+--------------------------------+------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |
|47770eb9100

**Вывод.** `orders` содержит 99 441 строку, и все поля прочитаны как `string`. Spark разбил файл на 5 разделов. Для идентификаторов строковый тип корректен, а временные отметки перед вычислениями нужно привести к `timestamp`.

### Практический блок 5.2. Явная схема для order_items

In [6]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

items = (
    spark.read
    .option("header", True)
    .schema(order_items_schema)
    .csv(str(STAGING / "order_items.csv"))
)

items.printSchema()
print("Строк:", items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Строк: 112650


**Вывод.** Явный `StructType` сразу сделал `order_item_id` целым, а `price` и `freight_value` — `double`, без отдельного прохода для вывода типов. Прочитано 112 650 позиций.

### Практический блок 5.3. Преобразование временных полей orders

In [7]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = orders_raw
for c in timestamp_cols:
    orders = orders.withColumn(
        c,
        F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
    )

orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



**Вывод.** После `to_timestamp` пять временных полей получили тип `timestamp`. Исходный `orders_raw` не изменился: `withColumn` создаёт новый DataFrame.

### Демонстрация lazy evaluation

Создание DataFrame из цепочки transformations занимает миллисекунды, потому что Spark лишь строит план.
Реальная работа происходит только при action (`count()`); `explain()` показывает физический план, который будет выполнен.

In [8]:
import time

t0 = time.perf_counter()
lazy_df = orders.filter(F.col("order_status") == "delivered").withColumn(
    "purchase_year", F.year("order_purchase_timestamp"))
t1 = time.perf_counter()
n = lazy_df.count()
t2 = time.perf_counter()

print(f"Построение плана (transformations): {(t1 - t0) * 1000:.1f} мс")
print(f"Выполнение count() (action):        {(t2 - t1) * 1000:.1f} мс, строк: {n}")
lazy_df.explain()

Построение плана (transformations): 21.2 мс
Выполнение count() (action):        185.0 мс, строк: 96478
== Physical Plan ==
*(1) Project [order_id#17, customer_id#18, order_status#19, order_purchase_timestamp#81, order_approved_at#82, order_delivered_carrier_date#83, order_delivered_customer_date#84, order_estimated_delivery_date#85, year(cast(order_purchase_timestamp#81 as date)) AS purchase_year#86]
+- *(1) Project [order_id#17, customer_id#18, order_status#19, gettimestamp(order_purchase_timestamp#20, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Europe/Moscow), true) AS order_purchase_timestamp#81, gettimestamp(order_approved_at#21, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Europe/Moscow), true) AS order_approved_at#82, gettimestamp(order_delivered_carrier_date#22, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Europe/Moscow), true) AS order_delivered_carrier_date#83, gettimestamp(order_delivered_customer_date#23, yyyy-MM-dd HH:mm:ss, Timest

**Вывод.** Цепочка `filter` + `withColumn` строится за миллисекунды, а время уходит на `count()`. В физическом плане фильтр по статусу передан в источник (`PushedFilters: EqualTo(order_status,delivered)`) — это оптимизация, которую даёт ленивое выполнение.

## 6. Основные преобразования DataFrame
### Практический блок 6.1. Фильтрация и вычисляемый признак стоимости позиции

In [9]:
expensive_items = (
    items
    .filter(F.col("price") >= 500)
    .withColumn(
        "gross_item_value",
        F.round(F.col("price") + F.col("freight_value"), 2)
    )
    .select(
        "order_id", "order_item_id",
        "price", "freight_value", "gross_item_value"
    )
)

print("Позиций с price >= 500:", expensive_items.count())

expensive_items.orderBy(
    F.desc("gross_item_value")
).show(5, truncate=False)

Позиций с price >= 500: 3239
+--------------------------------+-------------+------+-------------+----------------+
|order_id                        |order_item_id|price |freight_value|gross_item_value|
+--------------------------------+-------------+------+-------------+----------------+
|0812eb902a67711a1cb742b3cdaa65ae|1            |6735.0|194.31       |6929.31         |
|fefacc66af859508bf1a7934eab1e97f|1            |6729.0|193.21       |6922.21         |
|f5136e38d1a14a4dbd87dff67da82701|1            |6499.0|227.66       |6726.66         |
|a96610ab360d42a2e5335a3998b4718a|1            |4799.0|151.34       |4950.34         |
|199af31afc78c699f0dbf71fb178d4d4|1            |4690.0|74.34        |4764.34         |
+--------------------------------+-------------+------+-------------+----------------+
only showing top 5 rows


**Вывод.** Условию `price ≥ 500` соответствуют 3 239 из 112 650 позиций, как в методичке. Максимум — 6 735 + 194,31 = 6 929,31. `gross_item_value` относится к одной позиции, а не ко всему заказу: в одном `order_id` может быть несколько `order_item_id`.

### Практический блок 6.2. Временные преобразования заказов

In [10]:
delivered = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "is_late",
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)

with_delivery_date = delivered.filter(
    F.col("order_delivered_customer_date").isNotNull()
)
late = with_delivery_date.filter(F.col("is_late"))

n_with_date = with_delivery_date.count()
n_late = late.count()
print("Доставленных заказов:", delivered.count())
print("С известной датой доставки:", n_with_date)
print("Позже ожидаемой даты:", n_late, f"({n_late / n_with_date:.2%})")

Доставленных заказов: 96478
С известной датой доставки: 96470
Позже ожидаемой даты: 7826 (8.11%)


**Вывод.** Из 96 478 доставленных заказов у 96 470 есть фактическая дата доставки, из них 7 826 (8,11 %) доставлены позже ожидаемой даты. Результат совпадает с методичкой.

Дополнительно — `drop()`, `distinct()` и `dropDuplicates()`, которые также входят в перечень изучаемых transformations:

In [11]:
orders_short = orders.drop("order_approved_at", "order_delivered_carrier_date")
print("Столбцов до drop:", len(orders.columns), "после:", len(orders_short.columns))

print("Уникальных статусов (distinct):", orders.select("order_status").distinct().count())
print("Строк orders:", orders.count(),
      "| после dropDuplicates(['order_id']):", orders.dropDuplicates(["order_id"]).count())
print("Уникальных клиентов в orders:", orders.select("customer_id").distinct().count())

Столбцов до drop: 8 после: 6
Уникальных статусов (distinct): 8
Строк orders: 99441 | после dropDuplicates(['order_id']): 99441
Уникальных клиентов в orders: 99441


**Вывод.** `drop()` убрал 2 столбца; `distinct()` нашёл 8 статусов. Дубликатов `order_id` нет. Число уникальных `customer_id` равно числу заказов: в этой схеме `customer_id` выдаётся на каждый заказ, а постоянного покупателя обозначает `customer_unique_id`.

## 7. Работа с пропусками и преобразованием типов
### Практический блок 7.1. Подсчёт пропусков во временных полях orders

In [12]:
null_counts = orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in timestamp_cols
])

null_counts.show(vertical=True)

-RECORD 0-----------------------------
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



**Вывод.** Больше всего пропусков в `order_delivered_customer_date` — 2 965, а `purchase_timestamp` и `estimated_delivery_date` заполнены полностью. Удалять строки сразу нельзя: сначала нужно связать пропуски со статусом заказа.

### Практический блок 7.2. Предметная проверка пропусков по статусам

In [13]:
missing_delivery_by_status = (
    orders
    .filter(F.col("order_delivered_customer_date").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

missing_delivery_by_status.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



**Вывод.** Из 2 965 пропусков даты доставки только 8 приходятся на статус `delivered`. Остальные — shipped (1 107), canceled (619), unavailable (609) и промежуточные статусы. Значит, пропуск в основном означает, что доставки ещё не было или заказ отменён, а не что данные потеряны.

### Практический блок 7.3. Заполнение пропуска категориального признака

Таблица `products` читается с `inferSchema=True`, затем `fillna()` применяется только к `product_category_name`.
Пропуски физических характеристик только подсчитываются: в задаче выручки они не используются, и их заполнение
потребовало бы отдельного обоснования.

In [14]:
products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "products.csv"))
)

missing_categories = products.filter(
    F.col("product_category_name").isNull()
).count()

products_clean = products.fillna({
    "product_category_name": "unknown"
})

print("Товаров всего:", products.count())
print("Товаров без категории:", missing_categories)
products.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
]).show()

Товаров всего: 32951
Товаров без категории: 610
+----------------+-----------------+-----------------+----------------+
|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------------+-----------------+-----------------+----------------+
|               2|                2|                2|               2|
+----------------+-----------------+-----------------+----------------+



**Вывод.** У 610 из 32 951 товаров нет `product_category_name`. Значение `unknown` сохраняет их отдельной группой, и при соединении с позициями такие товары не теряются. Физические характеристики пропущены у 2 товаров; их не заполняем, потому что для выручки они не нужны.

## 8. Агрегации
### Практический блок 8.1. Распределение заказов по статусам

In [15]:
status_stats = (
    orders
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

status_stats.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



**Вывод.** Статус `delivered` преобладает: 96 478 из 99 441 заказа (97 %). Поэтому статистика доставки описывает в основном завершённые заказы, а по редким статусам (created — 5, approved — 2) проценты считать ненадёжно.

### Практический блок 8.2. Несколько агрегатных функций для способов оплаты

In [16]:
payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "order_payments.csv"))
)

payment_stats = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payments"),
        F.round(F.sum("payment_value"), 2).alias("total_value"),
        F.round(F.avg("payment_value"), 2).alias("avg_value")
    )
    .orderBy(F.desc("total_value"))
)

payment_stats.show(truncate=False)

+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



**Вывод.** Больше всего платежей картой (`credit_card`): 76 795 строк на ~12,54 млн (Spark выводит большие суммы в экспоненциальной записи, `1.254E7`). У `voucher` самый маленький средний чек — 65,70. Одна покупка может иметь несколько платёжных строк, поэтому `COUNT(*)` здесь — это число платежей, а не заказов.

## 9. Соединение DataFrame
### Практический блок 9.1. Позиции заказов + товары + перевод категорий

In [17]:
translations = (
    spark.read
    .option("header", True)
    .csv(str(STAGING / "product_category_name_translation.csv"))
)

sales = (
    items.alias("i")
    .join(
        products_clean.select(
            "product_id", "product_category_name"
        ).alias("p"),
        F.col("i.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        translations.alias("t"),
        F.col("p.product_category_name") ==
        F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("i.order_id").alias("order_id"),
        F.col("i.order_item_id").alias("order_item_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("i.price").alias("price"),
        F.col("i.freight_value").alias("freight_value"),
        F.coalesce(
            F.col("t.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("unknown")
        ).alias("category")
    )
)

print("Строк до JOIN:", items.count())
print("Строк после JOIN:", sales.count())
print(
    "Позиций категории unknown:",
    sales.filter(F.col("category") == "unknown").count()
)

# контроль: сколько позиций потерял бы INNER JOIN с products
inner_cnt = items.join(products_clean.select("product_id"), "product_id", "inner").count()
print("Строк при INNER JOIN с products:", inner_cnt)

Строк до JOIN: 112650
Строк после JOIN: 112650
Позиций категории unknown: 1603
Строк при INNER JOIN с products: 112650


**Вывод.** LEFT JOIN сохранил все 112 650 позиций. У 1 603 позиций категория `unknown` (товары без категории в справочнике). INNER JOIN с `products` тоже даёт 112 650 строк: для всех позиций товар в справочнике есть, неполна только сама категория. Проверять число строк обязательно: рост означал бы дубликаты ключа, уменьшение при INNER JOIN — потерю фактов.

### Практический блок 9.2. Заказы и клиенты, аналитика просрочки по штатам

In [18]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "customers.csv"))
)

late_by_state = (
    delivered
    .join(
        customers.select("customer_id", "customer_state"),
        on="customer_id",
        how="left"
    )
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered"),
        F.sum(
            F.when(F.col("is_late"), 1).otherwise(0)
        ).alias("late")
    )
    .withColumn(
        "late_pct",
        F.round(F.col("late") / F.col("delivered") * 100, 2)
    )
    .orderBy(F.desc("late_pct"))
)

late_by_state.show(10, truncate=False)

+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows


**Вывод.** Самая высокая доля опозданий у штата AL — 23,93 %, но там всего 397 доставок. У RJ доля ниже (13,47 %), но при 12 350 доставках это 1 664 опоздавших заказа — больше всех в абсолютных числах. Проценты нужно читать вместе с размером группы; причинной связи результат не доказывает.

## 10. Spark SQL
### Практический блок 10.1. Выручка по товарным категориям через SQL

In [19]:
sales.createOrReplaceTempView("sales")

category_sql = spark.sql("""
SELECT
    category,
    COUNT(*) AS items,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(price), 2) AS revenue,
    ROUND(AVG(price), 2) AS avg_price
FROM sales
GROUP BY category
ORDER BY revenue DESC
LIMIT 10
""")

category_sql.show(truncate=False)

+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+



**Вывод.** Больше всего по сумме цен у `health_beauty` (~1,259 млн), затем `watches_gifts` (~1,205 млн) и `bed_bath_table` (~1,037 млн), хотя по числу позиций лидирует `bed_bath_table` (11 115). Это сумма цен позиций, а не платёжная выручка: доставка не учтена.

### Практический блок 10.2. Эквивалентная агрегация через DataFrame API

In [20]:
category_sales = (
    sales
    .groupBy("category")
    .agg(
        F.count("*").alias("items"),
        F.countDistinct("order_id").alias("orders"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
)

print("Категорий в витрине:", category_sales.count())

category_sales.orderBy(
    F.desc("revenue")
).show(10, truncate=False)

# формальная проверка эквивалентности SQL и DataFrame API на топ-10
top10_api = category_sales.orderBy(F.desc("revenue")).limit(10)
print("Расхождений SQL vs DataFrame API:",
      category_sql.exceptAll(top10_api).count() + top10_api.exceptAll(category_sql).count())

Категорий в витрине: 74
+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows
Расхождений SQL vs DataFrame API: 0


**Вывод.** В витрине 74 группы: переведённые категории, непереведённые и `unknown`. `exceptAll` не нашёл ни одного расхождения между топ-10 через SQL и через DataFrame API: оба способа исполняются одним и тем же движком Spark SQL.

## 11. Запись результата
### Практический блок 11.1. Сохранение аналитической витрины в Parquet и CSV

In [21]:
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

(
    category_sales
    .write
    .mode("overwrite")
    .parquet(str(OUT / "category_sales_parquet"))
)

(
    category_sales
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(OUT / "category_sales_csv"))
)

category_back = spark.read.parquet(
    str(OUT / "category_sales_parquet")
)

print("Строк после round-trip:", category_back.count())
category_back.printSchema()

for d in ["category_sales_parquet", "category_sales_csv"]:
    print(d, "->", sorted(p.name for p in (OUT / d).iterdir()))

Строк после round-trip: 74
root
 |-- category: string (nullable = true)
 |-- items: long (nullable = true)
 |-- orders: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)

category_sales_parquet -> ['._SUCCESS.crc', '.part-00000-1e1e03eb-3fd1-4e54-bad3-aade34d565f9-c000.snappy.parquet.crc', '_SUCCESS', 'part-00000-1e1e03eb-3fd1-4e54-bad3-aade34d565f9-c000.snappy.parquet']
category_sales_csv -> ['._SUCCESS.crc', '.part-00000-281a49c7-b3c5-424d-b210-9c4022be896e-c000.csv.crc', '_SUCCESS', 'part-00000-281a49c7-b3c5-424d-b210-9c4022be896e-c000.csv']


**Вывод.** После записи и повторного чтения Parquet получены те же 74 строки, числовые типы сохранились (`long`, `double`), `inferSchema` не нужен. Каждый результат — это каталог: `part-*`-файл с данными, маркер `_SUCCESS` и контрольные `.crc`. Из-за `coalesce(1)` part-файл один.

### Практический блок 11.2. Корректное завершение SparkSession

`spark.stop()` освобождает ресурсы локального SparkContext и JVM. Это последняя ячейка учебной части;
в самостоятельной части создаётся новая сессия со своим именем приложения.

In [22]:
spark.stop()

# Часть II. Самостоятельная работа: распределённый анализ набора «Транзакции КОМУС»

## 1. Постановка задачи

**Источник данных.** Архив `Транзакции КОМУС.zip`, выданный в материалах практической работы №3 (учебный набор
транзакций розничной/оптовой сети канцелярских и художественных товаров). Набор используется только в учебных целях.
Покупатели в нём — юридические лица с условными (обезличенными) названиями и ИНН, данных физических лиц нет.

**Состав архива** (TSV, UTF-8 с BOM):

| Файл | Назначение | Роль |
|---|---|---|
| `Продажи.csv` | строки чеков: дата, чек, товар, филиал, количество, сумма, скидка, себестоимость, клиент, доставка | таблица фактов (~865 тыс. строк) |
| `Продукты.csv` | товар → название, бренд, товарная группа | справочник |
| `Клиенты.csv` | клиент → покупатель, юрлицо | справочник |
| `Суммы чеков.csv` | итоговые суммы чеков | агрегат (для сверки) |

**Цель распределённой обработки:** построить витрину продаж по товарным группам (выручка, себестоимость, валовая маржа,
число чеков и клиентов), оценить вклад филиалов и динамику по месяцам, а также проверить качество данных
(пропуски себестоимости, дубликаты в справочнике клиентов, аномальные скидки).

Требования п. 12.1 выполнены: основная таблица содержит 865 тыс. строк и 11 полей, числовые признаки — количество, сумма,
скидка, себестоимость; категориальные — филиал, товарная группа, бренд, клиент; есть пропуски (`NULL`) и строковые поля,
требующие преобразования типов (десятичная запятая, дата `dd.MM.yyyy`).

## 2. Архитектура входа: staging-этап

Spark не читает ZIP-архивы как источник, а имена файлов внутри архива записаны в кодировке CP866 (без UTF-8 флага).
Поэтому выбран воспроизводимый staging: архив распаковывается стандартным `zipfile` в каталог `komus_input` с
латинскими именами файлов. Содержимое файлов не меняется — Spark далее сам читает исходные TSV.
Такой способ проще JDBC/внешних драйверов и не зависит от ручной распаковки.

In [23]:
KOMUS_ZIP = Path("Транзакции КОМУС.zip")
KOMUS = Path("komus_input")
KOMUS.mkdir(exist_ok=True)

NAME_MAP = {
    "Продажи.csv": "sales.csv",
    "Продукты.csv": "products.csv",
    "Клиенты.csv": "clients.csv",
    "Суммы чеков.csv": "receipts.csv",
}

def zip_name(info):
    # без UTF-8 флага zipfile декодирует имя как CP437; исходная кодировка архива — CP866
    name = info.filename
    if not info.flag_bits & 0x800 and not any("а" <= ch.lower() <= "я" for ch in name):
        name = name.encode("cp437").decode("cp866")
    return name.split("/")[-1]

with zipfile.ZipFile(KOMUS_ZIP) as z:
    for info in z.infolist():
        name = zip_name(info)
        if name in NAME_MAP:
            (KOMUS / NAME_MAP[name]).write_bytes(z.read(info))
            print(f"{name:20s} -> {NAME_MAP[name]:14s} {info.file_size / 2**20:7.1f} MB")

Клиенты.csv          -> clients.csv        0.5 MB
Продажи.csv          -> sales.csv         89.0 MB
Продукты.csv         -> products.csv       3.3 MB
Суммы чеков.csv      -> receipts.csv      13.2 MB


**Вывод.** Staging распаковал 4 файла (продажи — 89 МБ). Имена файлов в архиве записаны в кодировке CP866, и без перекодировки получились бы нечитаемые имена — это одна из причин делать staging программно.

## 3. SparkSession и схема
### 3.1. Создание SparkSession и фиксация версий

In [24]:
with jvm_stderr_to_file():
    spark = (
        SparkSession.builder
        .appName("Practice3_Komus")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
# ERROR вместо WARN: при явной схеме Spark на каждом чтении предупреждает, что имена в заголовке CSV
# (кириллица) не совпадают с именами схемы (латиница) — это ожидаемо, переименование сделано намеренно
spark.sparkContext.setLogLevel("ERROR")

print("Python:", platform.python_version(), "| Java:", java_ver, "| Spark:", spark.version)
print("Master:", spark.sparkContext.master, "| App:", spark.sparkContext.appName,
      "| defaultParallelism:", spark.sparkContext.defaultParallelism)
for k in ["spark.driver.memory", "spark.sql.shuffle.partitions", "spark.ui.enabled"]:
    print(f"  {k} = {spark.conf.get(k)}")

Python: 3.13.2 | Java: openjdk version "21.0.6" 2025-01-21 | Spark: 4.2.1
Master: local[*] | App: Practice3_Komus | defaultParallelism: 8
  spark.driver.memory = 4g
  spark.sql.shuffle.partitions = 8
  spark.ui.enabled = false


### 3.2. Чтение основной таблицы строками

Первое чтение — без схемы: все поля приходят как `string`. Разделитель — табуляция, в данных встречаются
кавычки внутри значений (`ООО ""Положение""`), поэтому `escape` задан как `"`. BOM в начале файла Spark отбрасывает сам.

In [25]:
def tsv_reader(**opts):
    reader = (spark.read.option("header", True).option("sep", "\t")
              .option("encoding", "UTF-8").option("quote", '"').option("escape", '"'))
    for k, v in opts.items():
        reader = reader.option(k, v)
    return reader

def read_tsv(name, **opts):
    return tsv_reader(**opts).csv(str(KOMUS / name))

sales_raw = read_tsv("sales.csv")

print("Строк:", sales_raw.count())
print("Столбцов:", len(sales_raw.columns))
print("Столбцы:", sales_raw.columns)
print("Разделов:", sales_raw.rdd.getNumPartitions())
sales_raw.printSchema()
sales_raw.show(5, truncate=False)

Строк: 865222
Столбцов: 11
Столбцы: ['Себестоимость', 'Дата покупки', 'Номер чека', 'Product_key', 'Филиал', 'Количество', 'Сумма покупки', 'Сумма скидки', 'Client_ID', 'Машина доставки', 'Адрес']
Разделов: 8
root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: string (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Адрес: string (nullable = true)

+-------------+------------+----------+-----------+-----------------------------+----------+-------------+------------+---------+---------------+---------------------------------------------+
|Себестоимость|Дата покупки|Номер чека|Product_key|Филиал                       |Количество|Сумма покупки|Сумма скидки|Client

**Вывод.** Основная таблица — 865 222 строки, 11 столбцов, Spark читает её в 8 разделов. Без схемы все поля строковые. Видны проблемы, которые нужно решить: десятичная запятая (`113,33`), дата в формате `dd.MM.yyyy`, пропуски записаны строкой `NULL`, в названиях филиалов ведущие пробелы.

### 3.3. Явный StructType против inferSchema

Для таблицы фактов задаётся явная схема с латинскими именами столбцов. Пропуски в исходнике записаны строкой `NULL`,
поэтому указан `nullValue="NULL"`. Денежные поля и себестоимость оставлены строковыми: в них десятичная запятая
(`113,33`), которую CSV-ридер Spark не разбирает как `double` — их приведём явно на следующем шаге.
Для сравнения та же таблица читается с `inferSchema=True`.

In [26]:
sales_schema = StructType([
    StructField("cost_raw", StringType(), True),        # Себестоимость (десятичная запятая)
    StructField("purchase_date_raw", StringType(), True),  # Дата покупки dd.MM.yyyy
    StructField("receipt_no", StringType(), True),      # Номер чека
    StructField("product_key", StringType(), True),     # Product_key
    StructField("branch", StringType(), True),          # Филиал
    StructField("qty", IntegerType(), True),            # Количество
    StructField("amount_raw", StringType(), True),      # Сумма покупки
    StructField("discount_raw", StringType(), True),    # Сумма скидки
    StructField("client_id", StringType(), True),       # Client_ID
    StructField("delivery_car", StringType(), True),    # Машина доставки
    StructField("address", StringType(), True),         # Адрес доставки
])

t0 = time.perf_counter()
sales_str = tsv_reader(nullValue="NULL").schema(sales_schema).csv(str(KOMUS / "sales.csv"))
n_explicit = sales_str.count()
t_explicit = time.perf_counter() - t0

t0 = time.perf_counter()
sales_inferred = read_tsv("sales.csv", nullValue="NULL", inferSchema=True)
n_inferred = sales_inferred.count()
t_inferred = time.perf_counter() - t0

print(f"Явная схема:  {n_explicit} строк, чтение + count = {t_explicit:.2f} c")
print(f"inferSchema:  {n_inferred} строк, чтение + count = {t_inferred:.2f} c")
sales_str.printSchema()
sales_inferred.printSchema()

Явная схема:  865222 строк, чтение + count = 0.13 c
inferSchema:  865222 строк, чтение + count = 0.58 c
root
 |-- cost_raw: string (nullable = true)
 |-- purchase_date_raw: string (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- discount_raw: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- delivery_car: string (nullable = true)
 |-- address: string (nullable = true)

root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: integer (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = t

**Вывод.** `inferSchema` правильно определил только `Количество` (integer). Суммы и себестоимость остались строками из-за десятичной запятой, дата — строкой. При этом нужен дополнительный проход по файлу, поэтому чтение заметно медленнее, чем с явной схемой. Явный `StructType` задаёт латинские имена и тип `qty` и не тратит время на вывод типов.

## 4. Transformations
### 4.1. Приведение типов и вычисляемые признаки

* `trim(branch)` — в названиях филиалов есть ведущие пробелы (` ТЗ-1 ( ул.Фадеева)`), без очистки группы бы «раздвоились»;
* `regexp_replace(",", ".").cast("double")` — перевод десятичной запятой в число;
* `to_date(..., "dd.MM.yyyy")` — строковая дата в `DateType`, производные `month` и `weekday`;
* `margin = amount − cost` — валовая маржа строки (себестоимость в данных указана на всю строку, а не за единицу:
  например, 25 шт. за 250 руб. при себестоимости 113,33);
* `margin_pct`, `unit_price = amount / qty` (через `try_divide`: в Spark 4 по умолчанию включён ANSI-режим, и деление на ноль при нулевой сумме вызывает ошибку), `has_delivery` — признак доставки (заполнен адрес).

In [27]:
def to_num(col):
    return F.regexp_replace(F.col(col), ",", ".").cast("double")

sales = (
    sales_str
    .withColumn("branch", F.trim("branch"))
    .withColumn("purchase_date", F.to_date("purchase_date_raw", "dd.MM.yyyy"))
    .withColumn("cost", to_num("cost_raw"))
    .withColumn("amount", to_num("amount_raw"))
    .withColumn("discount", F.round(to_num("discount_raw"), 2))
    .drop("cost_raw", "purchase_date_raw", "amount_raw", "discount_raw")
    .withColumn("month", F.date_format("purchase_date", "yyyy-MM"))
    .withColumn("weekday", F.date_format("purchase_date", "E"))
    .withColumn("margin", F.round(F.col("amount") - F.col("cost"), 2))
    # в Spark 4 включён ANSI-режим: деление на 0 — ошибка, поэтому try_divide (amount = 0 / qty = 0 -> null)
    .withColumn("margin_pct", F.round(F.try_divide(F.col("margin"), F.col("amount")) * 100, 2))
    .withColumn("unit_price", F.round(F.try_divide(F.col("amount"), F.col("qty")), 2))
    .withColumn("has_delivery", F.col("address").isNotNull())
)

sales.printSchema()
sales.select("purchase_date", "month", "branch", "product_key", "qty",
             "amount", "cost", "margin", "margin_pct", "unit_price").show(5)
print("Период данных:", sales.agg(F.min("purchase_date"), F.max("purchase_date")).first())

root
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- client_id: string (nullable = true)
 |-- delivery_car: string (nullable = true)
 |-- address: string (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- cost: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- month: string (nullable = true)
 |-- weekday: string (nullable = true)
 |-- margin: double (nullable = true)
 |-- margin_pct: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- has_delivery: boolean (nullable = false)

+-------------+-------+--------------------+-----------+---+------+------+-------+----------+----------+
|purchase_date|  month|              branch|product_key|qty|amount|  cost| margin|margin_pct|unit_price|
+-------------+-------+--------------------+-----------+---+------+------+-------+----------+-------

**Вывод.** После приведения типов суммы стали `double`, дата — `date`. Добавлены признаки: месяц, день недели, маржа, маржа в %, цена за единицу, наличие доставки. Данные охватывают период с 16.06.2020 по 31.12.2021. Пример: 25 шт. на 250 руб. при себестоимости 113,33 дают маржу 136,67 (54,7 %).

### 4.2. select / filter / distinct / dropDuplicates / orderBy

In [28]:
print("Филиалы (distinct):")
sales.select("branch").distinct().orderBy("branch").show(truncate=False)

n_all = sales.count()
n_dedup = sales.dropDuplicates().count()
print(f"Строк всего: {n_all}, после dropDuplicates() по всем полям: {n_dedup}, полных дублей: {n_all - n_dedup}")

zero_amount = sales.filter(F.col("amount") == 0)
neg_discount = sales.filter(F.col("discount") < 0)
neg_margin = sales.filter(F.col("margin") < 0)
print("Строк с нулевой суммой:", zero_amount.count())
print("Строк с отрицательной скидкой:", neg_discount.count())
print("Строк с отрицательной маржой (продажа ниже себестоимости):", neg_margin.count())

big_lines = (
    sales
    .filter(F.col("amount") >= 10000)
    .select("purchase_date", "branch", "receipt_no", "product_key", "qty", "amount", "margin")
    .orderBy(F.desc("amount"))
)
print("Строк чека на сумму ≥ 10 000:", big_lines.count())
big_lines.show(5, truncate=False)

Филиалы (distinct):
+----------------------------+
|branch                      |
+----------------------------+
|Багетная мастерская         |
|Кафе-1 ( ул.Фадеева)        |
|РС (Распределительный склад)|
|ТЗ-1 ( ул.Фадеева)          |
|ТЗ-2 ( Ленинградское шоссе) |
|ТЗ-3 (Курский)              |
+----------------------------+

Строк всего: 865222, после dropDuplicates() по всем полям: 865222, полных дублей: 0
Строк с нулевой суммой: 67
Строк с отрицательной скидкой: 39
Строк с отрицательной маржой (продажа ниже себестоимости): 0
Строк чека на сумму ≥ 10 000: 284
+-------------+---------------------------+----------+-----------+---+--------+---------+
|purchase_date|branch                     |receipt_no|product_key|qty|amount  |margin   |
+-------------+---------------------------+----------+-----------+---+--------+---------+
|2021-10-15   |ТЗ-2 ( Ленинградское шоссе)|NULL      |6736_20_289|50 |231500.0|128740.94|
|2021-12-03   |ТЗ-2 ( Ленинградское шоссе)|NULL      |8044_20_278|50 

**Вывод.** В данных 6 филиалов (после `trim`), полных дубликатов строк нет. Найдены аномалии: 67 строк с нулевой суммой (без `try_divide` они давали деление на ноль) и 39 строк с отрицательной скидкой. Продаж ниже себестоимости нет. Крупнейшая строка — 231 500 руб. за 50 шт. в ТЗ-2.

## 5. Качество данных
### 5.1. Подсчёт пропусков

In [29]:
null_cols = ["cost", "qty", "receipt_no", "delivery_car", "address", "amount", "discount", "purchase_date"]
sales_nulls = sales.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in null_cols])
sales_nulls.show(vertical=True)

-RECORD 0---------------
 cost          | 86431  
 qty           | 100    
 receipt_no    | 3934   
 delivery_car  | 832181 
 address       | 832181 
 amount        | 0      
 discount      | 0      
 purchase_date | 0      



### 5.2. Предметная диагностика пропусков по филиалам

In [30]:
null_by_branch = (
    sales.groupBy("branch")
    .agg(
        F.count("*").alias("lines"),
        F.sum(F.col("cost").isNull().cast("int")).alias("cost_null"),
        F.sum(F.col("receipt_no").isNull().cast("int")).alias("receipt_null"),
        F.sum(F.col("has_delivery").cast("int")).alias("with_address"),
        F.sum(F.col("qty").isNull().cast("int")).alias("qty_null"),
    )
    .withColumn("cost_null_pct", F.round(F.col("cost_null") / F.col("lines") * 100, 2))
    .orderBy(F.desc("lines"))
)
null_by_branch.show(truncate=False)

print("Строки без количества:")
sales.filter(F.col("qty").isNull()).select(
    "purchase_date", "branch", "product_key", "amount", "cost", "unit_price").show(5)

+----------------------------+------+---------+------------+------------+--------+-------------+
|branch                      |lines |cost_null|receipt_null|with_address|qty_null|cost_null_pct|
+----------------------------+------+---------+------------+------------+--------+-------------+
|ТЗ-2 ( Ленинградское шоссе) |431372|42858    |2807        |8379        |43      |9.94         |
|ТЗ-1 ( ул.Фадеева)          |238046|23799    |19          |19201       |33      |10.0         |
|ТЗ-3 (Курский)              |190730|19243    |11          |5212        |24      |10.09        |
|Кафе-1 ( ул.Фадеева)        |3977  |417      |0           |0           |0       |10.49        |
|Багетная мастерская         |848   |91       |848         |0           |0       |10.73        |
|РС (Распределительный склад)|249   |23       |249         |249         |0       |9.24         |
+----------------------------+------+---------+------------+------------+--------+-------------+

Строки без количества:
+-----

**Вывод.** Себестоимость пропущена примерно в 10 % строк каждого филиала. Доля почти одинаковая везде, значит пропуски не связаны с конкретным филиалом. У Багетной мастерской и РС номер чека отсутствует во всех строках, а у РС адрес доставки есть во всех строках — это оптовые отгрузки. Пустой адрес у 832 181 строки означает продажу без доставки. Количество пропущено в 100 строках, при этом сумма в них известна.

### 5.3. Обоснованная обработка пропусков

| Поле | Решение | Предметная причина |
|---|---|---|
| `address`, `delivery_car` | не удалять; `fillna("нет доставки")`, тип продажи — в `delivery_type` | пусто у всех продаж без доставки (покупка в магазине) — это состояние процесса, а не потеря данных |
| `cost` | **не** заполнять нулём, строки не удалять; признак `cost_known`; маржа считается только по строкам с известной себестоимостью | ноль завысил бы маржу до 100 %, удаление занизило бы выручку |
| `qty` | не заполнять; строки остаются в выручке, `unit_price` для них = null | количество нельзя восстановить надёжно, а сумма продажи известна |
| `receipt_no` | `fillna("NO_RECEIPT")` в отдельном поле `receipt_label`; составной ключ чека `receipt_key` для таких строк = null и в `countDistinct` не участвует | продажи без номера чека (в т.ч. отгрузки со склада) — реальные операции |

In [31]:
sales_clean = (
    sales
    .withColumn("delivery_type", F.when(F.col("has_delivery"), "Доставка").otherwise("Без доставки"))
    .withColumn("cost_known", F.col("cost").isNotNull())
    .fillna({"address": "нет доставки", "delivery_car": "нет доставки"})
    .withColumn("receipt_label", F.coalesce(F.col("receipt_no"), F.lit("NO_RECEIPT")))
    # номер чека уникален только внутри филиала и даты (см. п. 6.5), поэтому ключ чека составной
    .withColumn("receipt_key", F.when(F.col("receipt_no").isNotNull(),
                                      F.concat_ws("|", "branch", F.col("purchase_date").cast("string"), "receipt_no")))
)

print("Строк до обработки:", sales.count(), "| после:", sales_clean.count())
sales_clean.groupBy("delivery_type").agg(
    F.count("*").alias("lines"),
    F.round(F.sum("amount"), 2).alias("revenue")
).show()
sales_clean.groupBy("cost_known").agg(
    F.count("*").alias("lines"),
    F.round(F.sum("amount"), 2).alias("revenue")
).show()

Строк до обработки: 865222 | после: 865222
+-------------+------+--------------+
|delivery_type| lines|       revenue|
+-------------+------+--------------+
|     Доставка| 33041|  2.58573526E7|
| Без доставки|832181|1.0588121274E8|
+-------------+------+--------------+

+----------+------+--------------+
|cost_known| lines|       revenue|
+----------+------+--------------+
|      true|778791|1.1847487535E8|
|     false| 86431| 1.326368999E7|
+----------+------+--------------+



**Вывод.** Ни одна строка не удалена (865 222 до и после). Доставка — всего 3,8 % строк, но 25,9 млн руб. (19,6 % выручки). Себестоимость известна для строк на 118,5 млн из 131,7 млн выручки (89,9 %), поэтому маржу считаем только по ним.

## 6. Агрегации и JOIN
### 6.1. Группировка по филиалам (несколько агрегатных функций)

In [32]:
branch_stats = (
    sales_clean.groupBy("branch")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_key").alias("receipts"),
        F.countDistinct("client_id").alias("clients"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.avg("amount"), 2).alias("avg_line"),
        F.round(F.sum(F.when(F.col("cost_known"), F.col("margin"))), 2).alias("margin"),
        F.round(F.sum(F.when(F.col("cost_known"), F.col("amount"))), 2).alias("revenue_cost_known"),
    )
    .withColumn("margin_pct", F.round(F.try_divide(F.col("margin"), F.col("revenue_cost_known")) * 100, 2))
    .withColumn("avg_receipt", F.round(F.try_divide(F.col("revenue"), F.col("receipts")), 2))
    .drop("revenue_cost_known")
    .orderBy(F.desc("revenue"))
)
branch_stats.show(truncate=False)

+----------------------------+------+--------+-------+-------------+--------+-------------+----------+-----------+
|branch                      |lines |receipts|clients|revenue      |avg_line|margin       |margin_pct|avg_receipt|
+----------------------------+------+--------+-------+-------------+--------+-------------+----------+-----------+
|ТЗ-2 ( Ленинградское шоссе) |431372|78784   |406    |5.875080324E7|136.2   |2.64416012E7 |50.06     |745.72     |
|ТЗ-1 ( ул.Фадеева)          |238046|46337   |1469   |4.434316832E7|186.28  |1.991734847E7|49.94     |956.97     |
|ТЗ-3 (Курский)              |190730|32296   |303    |2.624905561E7|137.62  |1.183914549E7|50.21     |812.76     |
|Багетная мастерская         |848   |0       |1      |1399096.07   |1649.88 |636788.78    |50.59     |NULL       |
|РС (Распределительный склад)|249   |0       |9      |788679.75    |3167.39 |354210.08    |47.76     |NULL       |
|Кафе-1 ( ул.Фадеева)        |3977  |2459    |2      |207762.35    |52.24   |938

**Вывод.** Больше всего выручки у ТЗ-2 — 58,75 млн (44,6 %), затем ТЗ-1 — 44,34 млн и ТЗ-3 — 26,25 млн. У ТЗ-1 больше всего клиентов (1 469) и самый большой средний чек (957 руб.). Маржа почти одинаковая во всех филиалах (~50 %). У Багетной мастерской и РС средний чек не рассчитан: номеров чеков в их строках нет.

### 6.2. Динамика по месяцам

In [33]:
monthly = (
    sales_clean.groupBy("month")
    .agg(
        F.countDistinct("receipt_key").alias("receipts"),
        F.round(F.sum("amount"), 0).alias("revenue"),
        F.round(F.sum("discount"), 0).alias("discount"),
    )
    .orderBy("month")
)
monthly.show(30)

+-------+--------+-----------+--------+
|  month|receipts|    revenue|discount|
+-------+--------+-----------+--------+
|2020-06|     662|   269718.0| 57215.0|
|2020-07|    2438|  1317946.0|208151.0|
|2020-08|    2254|  1309761.0| 15661.0|
|2020-09|    3646|  2121197.0|     0.0|
|2020-10|    4173|  2684683.0| 20036.0|
|2020-11|    3982|  2245581.0|  1824.0|
|2020-12|    4718|  3386536.0|    63.0|
|2021-01|    3653|  2562761.0|  7761.0|
|2021-02|    4471|  2972448.0|     7.0|
|2021-03|    5030|  2829780.0|   147.0|
|2021-04|    4879|  2542106.0|   163.0|
|2021-05|    6235|  3534212.0|   167.0|
|2021-06|    6254|  4308662.0| 14207.0|
|2021-07|   11671|1.1193031E7|130975.0|
|2021-08|   12004| 1.163173E7|210065.0|
|2021-09|   19273|1.9053263E7|434490.0|
|2021-10|   20566|1.7447679E7|314085.0|
|2021-11|   19954|1.7303145E7|351826.0|
|2021-12|   24013|2.3024329E7|509724.0|
+-------+--------+-----------+--------+



**Вывод.** Выручка выросла с 4,3 млн в июне 2021 до 23,0 млн в декабре 2021. Рост во многом объясняется тем, что в данных появились новые филиалы: ТЗ-1 — с января 2021, ТЗ-3 — с апреля, Багетная мастерская — с июля; до 2021 года продажи есть только по ТЗ-2 и РС. Скидки были крупными в июне–июле 2020 и со второй половины 2021, а между этими периодами почти отсутствовали.

### 6.3. LEFT JOIN и INNER JOIN со справочником товаров

In [34]:
products_k = read_tsv("products.csv").toDF("product_key", "product_name", "brand", "product_group")
print("Товаров в справочнике:", products_k.count(),
      "| уникальных ключей:", products_k.select("product_key").distinct().count())

n_before = sales_clean.count()
sales_left = sales_clean.join(products_k, on="product_key", how="left")
sales_inner = sales_clean.join(products_k, on="product_key", how="inner")
n_left, n_inner = sales_left.count(), sales_inner.count()
n_no_match = sales_clean.join(products_k, on="product_key", how="left_anti").count()

print(f"Строк до JOIN:        {n_before}")
print(f"После LEFT JOIN:      {n_left}")
print(f"После INNER JOIN:     {n_inner}")
print(f"Без пары (left_anti): {n_no_match}")
print("Пропуски после LEFT JOIN:",
      sales_left.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                         for c in ["product_name", "brand", "product_group"]]).first().asDict())

Товаров в справочнике: 21163 | уникальных ключей: 21163
Строк до JOIN:        865222
После LEFT JOIN:      865222
После INNER JOIN:     865222
Без пары (left_anti): 0
Пропуски после LEFT JOIN: {'product_name': 0, 'brand': 27, 'product_group': 0}


**Вывод.** Справочник товаров уникален по ключу (21 163 ключа), и у каждой продажи есть пара. Поэтому LEFT и INNER JOIN дают одинаковые 865 222 строки, а `left_anti` — 0. Для витрины всё равно выбран LEFT JOIN: если справочник в будущем окажется неполным, продажи не потеряются. После JOIN бренд не заполнен в 27 строках.

### 6.4. Контроль строк при JOIN со справочником клиентов

Справочник клиентов не уникален по `Client_ID`: у одного клиента может быть несколько юрлиц (`ЮРЛИЦО №1`, `№3`…).
Наивный JOIN размножил бы строки продаж — это и проверяется сравнением числа строк.

In [35]:
clients_raw = read_tsv("clients.csv").toDF("client_id", "buyer", "contractor_name")
print("Строк в справочнике клиентов:", clients_raw.count(),
      "| уникальных client_id:", clients_raw.select("client_id").distinct().count())

naive = sales_left.join(clients_raw, on="client_id", how="left")
print("LEFT JOIN без дедупликации:", naive.count(), "строк (было", n_left, ")")

clients = (clients_raw
           .groupBy("client_id")
           .agg(F.first("buyer").alias("buyer"),
                F.countDistinct("contractor_name").alias("legal_entities")))

sales_full = sales_left.join(clients, on="client_id", how="left")
print("LEFT JOIN после агрегации справочника до client_id:", sales_full.count(), "строк")
print("Клиентов продаж без пары в справочнике:",
      sales_left.select("client_id").distinct().join(clients, "client_id", "left_anti").count())

Строк в справочнике клиентов: 3838 | уникальных client_id: 3227
LEFT JOIN без дедупликации: 874573 строк (было 865222 )
LEFT JOIN после агрегации справочника до client_id: 865222 строк
Клиентов продаж без пары в справочнике: 0


**Вывод.** В справочнике клиентов 3 838 строк, но только 3 227 уникальных `client_id`. Наивный LEFT JOIN увеличил таблицу продаж с 865 222 до 874 573 строк (+9 351), и выручка оказалась бы завышена. После того как справочник свёрнут до одной строки на клиента, число строк снова 865 222. Это главный практический пример того, зачем сравнивать число строк до и после JOIN.

### 6.5. INNER JOIN: сверка строк продаж с итогами чеков

Таблица `Суммы чеков.csv` хранит итог каждого чека. INNER JOIN агрегата продаж по чеку с этой таблицей — естественная
сверка: в результат попадают только чеки, присутствующие в обоих источниках, и для них суммы должны совпасть.
Сначала проверяется ключ: номер чека повторяется в разных филиалах и днях, поэтому соединение выполняется по
составному ключу (филиал, дата, номер чека).

In [36]:
receipts = (
    read_tsv("receipts.csv", nullValue="NULL")
    .toDF("purchase_date_raw", "client_id", "address", "delivery_car", "branch", "receipt_no", "receipt_total_raw")
    .withColumn("branch", F.trim("branch"))
    .withColumn("purchase_date", F.to_date("purchase_date_raw", "dd.MM.yyyy"))
    .withColumn("receipt_total", to_num("receipt_total_raw"))
    .filter(F.col("receipt_no").isNotNull())   # чеки без номера невозможно сопоставить по ключу
    .select("branch", "purchase_date", "receipt_no", "receipt_total")
)

key = ["branch", "purchase_date", "receipt_no"]
print("Строк в таблице чеков (с номером):", receipts.count())
print("Уникальных номеров чеков:", receipts.select("receipt_no").distinct().count())
print("Уникальных составных ключей:", receipts.select(*key).distinct().count())

sales_by_receipt = (
    sales_clean.filter(F.col("receipt_no").isNotNull())
    .groupBy(*key)
    .agg(F.count("*").alias("lines"), F.round(F.sum("amount"), 2).alias("lines_total"))
)
check = sales_by_receipt.join(receipts, on=key, how="inner")
n_sr, n_r, n_check = sales_by_receipt.count(), receipts.count(), check.count()
print(f"Чеков в продажах: {n_sr} | в таблице чеков: {n_r} | после INNER JOIN: {n_check}")
print("Чеков с расхождением суммы > 0.01:",
      check.filter(F.abs(F.col("lines_total") - F.col("receipt_total")) > 0.01).count())
check.agg(F.round(F.sum("lines_total"), 2).alias("sum_lines"),
          F.round(F.sum("receipt_total"), 2).alias("sum_receipts")).show()

Строк в таблице чеков (с номером): 159876
Уникальных номеров чеков: 81837
Уникальных составных ключей: 159876
Чеков в продажах: 159876 | в таблице чеков: 159876 | после INNER JOIN: 159876
Чеков с расхождением суммы > 0.01: 0
+--------------+--------------+
|     sum_lines|  sum_receipts|
+--------------+--------------+
|1.2329758915E8|1.2329758877E8|
+--------------+--------------+



**Вывод.** Номер чека не уникален: 159 876 чеков с номером приходятся всего на 81 837 разных номеров. Уникален только составной ключ (филиал, дата, номер). По нему INNER JOIN сопоставил все 159 876 чеков, и ни в одном сумма строк не расходится с итогом чека больше чем на 0,01. Общая разница — 0,38 руб. на 123,3 млн (округление). Поэтому дальше чеки считаются по составному ключу `receipt_key`; если считать по одному номеру, число чеков, например в июле 2021, получилось бы на 19 % меньше.

## 7. Spark SQL

In [37]:
sales_full.createOrReplaceTempView("komus_sales")

top_groups_sql = spark.sql("""
SELECT
    product_group,
    COUNT(*)                                   AS lines,
    COUNT(DISTINCT receipt_key)                AS receipts,
    ROUND(SUM(amount), 2)                      AS revenue,
    ROUND(try_divide(SUM(CASE WHEN cost_known THEN margin END),
                     SUM(CASE WHEN cost_known THEN amount END)) * 100, 2) AS margin_pct
FROM komus_sales
GROUP BY product_group
ORDER BY revenue DESC
LIMIT 10
""")
top_groups_sql.show(truncate=False)

+-------------------------------------+-----+--------+----------+----------+
|product_group                        |lines|receipts|revenue   |margin_pct|
+-------------------------------------+-----+--------+----------+----------+
|Основы-холст на подрамнике-Базовый   |6703 |4757    |4407000.88|50.96     |
|z-Уценка                             |37800|16361   |3296317.47|50.1      |
|Живопись-акварель-наборы             |6582 |6204    |3182778.15|50.32     |
|Бумага-Картон-пенокартон             |2531 |2223    |2716844.79|49.76     |
|Подрамники, рамы-Планшеты            |3850 |3622    |2550475.59|50.39     |
|Живопись-масло-профи-Мастер-Класс ЗХК|15925|5368    |2367607.02|50.41     |
|Сумки-на молнии ткань/кожа           |2949 |2867    |2184469.42|50.52     |
|Кисти-Масло-Синтетика                |24409|12439   |2103641.02|49.93     |
|Графические материалы-маркеры-маркеры|15459|7155    |1948429.28|49.96     |
|Мольберты-Этюдники                   |593  |589     |1855947.64|50.0      |

**Вывод.** Больше всего выручки у группы «Основы-холст на подрамнике-Базовый» (4,41 млн), на втором месте уценённые товары «z-Уценка» (3,30 млн, но больше всего строк — 37 800), на третьем акварельные наборы (3,18 млн). По числу строк лидируют кисти и маркеры — дешёвые товары, которые покупают часто.

In [38]:
brands_sql = spark.sql("""
SELECT
    brand,
    COUNT(DISTINCT product_key)                AS products,
    COUNT(DISTINCT client_id)                  AS clients,
    ROUND(SUM(amount), 0)                      AS revenue,
    ROUND(SUM(discount), 0)                    AS discount
FROM komus_sales
WHERE brand IS NOT NULL
GROUP BY brand
HAVING SUM(amount) > 1000000
ORDER BY revenue DESC
""")
brands_sql.show(10, truncate=False)
print("Брендов с выручкой > 1 млн:", brands_sql.count())

branch_month_sql = spark.sql("""
SELECT branch, month, revenue,
       ROUND(revenue / SUM(revenue) OVER (PARTITION BY month) * 100, 1) AS share_pct
FROM (
    SELECT branch, month, ROUND(SUM(amount), 0) AS revenue
    FROM komus_sales
    WHERE month IN ('2020-12', '2021-12')
    GROUP BY branch, month
)
ORDER BY month, revenue DESC
""")
branch_month_sql.show(truncate=False)

+----------------+--------+-------+-----------+--------+
|brand           |products|clients|revenue    |discount|
+----------------+--------+-------+-----------+--------+
|ЗХК             |808     |1088   |1.6043042E7|322203.0|
|Гамма           |1035    |1111   |1.219573E7 |211124.0|
|Faber-Castell   |881     |625    |7086985.0  |101487.0|
|ХМ              |414     |378    |6100526.0  |92631.0 |
|Ferrario        |1335    |333    |5216057.0  |89658.0 |
|Maimeri         |629     |279    |3709149.0  |73396.0 |
|Туюкан          |170     |199    |3601287.0  |91302.0 |
|Daler-rowney    |350     |450    |3498913.0  |67131.0 |
|Живописные кисти|405     |440    |3062930.0  |54094.0 |
|СПБФ Гознака    |122     |649    |3037966.0  |49099.0 |
+----------------+--------+-------+-----------+--------+
only showing top 10 rows
Брендов с выручкой > 1 млн: 31
+----------------------------+-------+---------+---------+
|branch                      |month  |revenue  |share_pct|
+---------------------------

**Вывод.** Больше всего выручки у брендов ЗХК (16,0 млн) и «Гамма» (12,2 млн); у «Гаммы» больше всего клиентов (1 111). Всего 31 бренд набрал больше 1 млн. Оконная функция показывает, как менялась структура сети: в декабре 2020 89 % выручки давал ТЗ-2, а в декабре 2021 впереди уже ТЗ-1 (42,7 %), у ТЗ-2 — 32,6 %.

### 7.1. Сопоставление SQL и DataFrame API

In [39]:
top_groups_api = (
    sales_full.groupBy("product_group")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_key").alias("receipts"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.try_divide(F.sum(F.when(F.col("cost_known"), F.col("margin"))),
                             F.sum(F.when(F.col("cost_known"), F.col("amount")))) * 100, 2).alias("margin_pct"),
    )
    .orderBy(F.desc("revenue"))
    .limit(10)
)
diff = top_groups_sql.exceptAll(top_groups_api).count() + top_groups_api.exceptAll(top_groups_sql).count()
print("Расхождений SQL vs DataFrame API:", diff)

Расхождений SQL vs DataFrame API: 0


**Вывод.** SQL-запрос и цепочка `groupBy().agg()` дали одинаковый топ-10: расхождений 0. Оба способа строят план для одного и того же оптимизатора Catalyst.

## 8. Экспорт: итоговая витрина по товарным группам

Итоговый аналитический DataFrame — витрина на уровне товарной группы. Пишется в Parquet (основной формат)
и, так как агрегат небольшой, дополнительно в один CSV-файл через `coalesce(1)`.

In [40]:
group_mart = (
    sales_full.groupBy("product_group")
    .agg(
        F.countDistinct("product_key").alias("products"),
        F.countDistinct("brand").alias("brands"),
        F.count("*").alias("lines"),
        F.countDistinct("receipt_key").alias("receipts"),
        F.countDistinct("client_id").alias("clients"),
        F.sum("qty").alias("qty"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.sum("discount"), 2).alias("discount"),
        F.round(F.sum(F.when(F.col("cost_known"), F.col("amount"))), 2).alias("revenue_cost_known"),
        F.round(F.sum(F.when(F.col("cost_known"), F.col("margin"))), 2).alias("margin"),
    )
    .withColumn("margin_pct", F.round(F.try_divide(F.col("margin"), F.col("revenue_cost_known")) * 100, 2))
    .withColumn("cost_coverage_pct", F.round(F.try_divide(F.col("revenue_cost_known"), F.col("revenue")) * 100, 2))
    .orderBy(F.desc("revenue"))
)

n_mart = group_mart.count()
print("Строк в витрине (товарных групп):", n_mart)
group_mart.show(5, truncate=False)

KOUT = Path("spark_output_komus")
KOUT.mkdir(exist_ok=True)
group_mart.write.mode("overwrite").parquet(str(KOUT / "group_mart_parquet"))
group_mart.coalesce(1).write.mode("overwrite").option("header", True).csv(str(KOUT / "group_mart_csv"))

for d in ["group_mart_parquet", "group_mart_csv"]:
    print(d, "->", sorted(p.name for p in (KOUT / d).iterdir()))

Строк в витрине (товарных групп): 437
+----------------------------------+--------+------+-----+--------+-------+-----+----------+---------+------------------+----------+----------+-----------------+
|product_group                     |products|brands|lines|receipts|clients|qty  |revenue   |discount |revenue_cost_known|margin    |margin_pct|cost_coverage_pct|
+----------------------------------+--------+------+-----+--------+-------+-----+----------+---------+------------------+----------+----------+-----------------+
|Основы-холст на подрамнике-Базовый|223     |3     |6703 |4757    |211    |10516|4407000.88|111558.54|3967948.3         |2022021.64|50.96     |90.04            |
|z-Уценка                          |1348    |25    |37800|16361   |355    |69821|3296317.47|50435.77 |2950209.99        |1478031.01|50.1      |89.5             |
|Живопись-акварель-наборы          |69      |10    |6582 |6204    |164    |8688 |3182778.15|45024.92 |2876068.59        |1447107.11|50.32     |90.36    

### 8.1. Round-trip: повторное чтение Parquet

In [41]:
mart_back = spark.read.parquet(str(KOUT / "group_mart_parquet"))
print("Строк до записи:", n_mart, "| после чтения:", mart_back.count())
same_types = [(f.name, f.dataType) for f in mart_back.schema] == [(f.name, f.dataType) for f in group_mart.schema]
print("Имена и типы столбцов совпадают:", same_types)
print("Различия в nullable (столбец, до записи, после чтения):",
      [(f.name, g.nullable, f.nullable) for f, g in zip(mart_back.schema, group_mart.schema) if f.nullable != g.nullable])
mart_back.printSchema()

Строк до записи: 437 | после чтения: 437
Имена и типы столбцов совпадают: True
Различия в nullable (столбец, до записи, после чтения): [('products', False, True), ('brands', False, True), ('lines', False, True), ('receipts', False, True), ('clients', False, True)]
root
 |-- product_group: string (nullable = true)
 |-- products: long (nullable = true)
 |-- brands: long (nullable = true)
 |-- lines: long (nullable = true)
 |-- receipts: long (nullable = true)
 |-- clients: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue_cost_known: double (nullable = true)
 |-- margin: double (nullable = true)
 |-- margin_pct: double (nullable = true)
 |-- cost_coverage_pct: double (nullable = true)



**Вывод.** Итоговая витрина — 437 товарных групп. Она записана в Parquet и CSV, каждый в виде каталога с part-файлом. После повторного чтения Parquet число строк (437), имена и типы столбцов совпали. Отличается только признак `nullable` у счётчиков: при записи Parquet Spark помечает все столбцы как допускающие null. Данные это не меняет.

In [42]:
spark.stop()

## 9. Итоги самостоятельной работы

**Итоговые выводы**

1. Набор «Транзакции КОМУС» — это 865 222 строки продаж за 16.06.2020–31.12.2021 на 131,7 млн руб. в 6 подразделениях. Почти вся выручка приходится на три торговых зала: ТЗ-2 — 44,6 %, ТЗ-1 — 33,7 %, ТЗ-3 — 19,9 %.
2. Перед анализом данные пришлось привести к нужным типам: заменить десятичную запятую, разобрать даты `dd.MM.yyyy`, учесть `NULL` как пропуск, убрать пробелы в названиях филиалов. `inferSchema` с этим не справляется, поэтому для основной таблицы нужна явная схема.
3. Пропуски в основном объясняются бизнес-процессом: пустой адрес означает продажу без доставки, отсутствие номера чека — оптовую отгрузку или работу мастерской. Себестоимость пропущена равномерно примерно в 10 % строк, поэтому её не заполняли, а маржу считали по 89,9 % выручки, где себестоимость известна.
4. JOIN проверялись по числу строк, и это помогло найти две ошибки, которые иначе прошли бы незаметно: дубликаты `client_id` в справочнике клиентов (+9 351 строка при наивном JOIN) и неуникальность номера чека (уникален только ключ «филиал + дата + номер»).
5. Строки продаж полностью совпадают с отдельной таблицей итогов чеков (159 876 чеков, расхождений нет) — данные внутренне согласованы.
6. Рост выручки в 2021 году во многом связан с тем, что в данных появились новые филиалы (ТЗ-1 — с января, ТЗ-3 — с апреля), поэтому сравнивать месяцы по всей сети нужно с учётом того, какие филиалы в них есть.
7. Ассортимент — художественные материалы. Лидеры по выручке — холсты на подрамниках, уценка и акварельные наборы; бренды — ЗХК и «Гамма».

**Ограничения анализа**

1. Маржа почти везде около 50 % — по филиалам, группам и брендам. Скорее всего, себестоимость в учебном наборе сгенерирована как доля от цены, поэтому сравнивать прибыльность по марже нельзя.
2. Что именно означают поля, не задокументировано: включает ли «Сумма покупки» скидку, откуда берутся отрицательные скидки (39 строк) и нулевые суммы (67 строк). Выводы о скидках поэтому только описательные.
3. Периоды работы филиалов в данных разные, так что сравнение их годовых показателей смещено.
4. Расчёт выполнялся в режиме `local[*]` на одном компьютере: время выполнения не говорит о том, как код будет работать на реальном кластере, хотя сам код переносится без изменений логики.

# Ответы на контрольные вопросы

**1. Чем `local[*]` отличается от физического кластера и что сохраняется?**
`local[*]` запускает Driver и вычисления в одной JVM на одном компьютере, а «исполнители» — это потоки по числу логических
ядер. Нет сети, распределённого хранения и отказов узлов. Сохраняется программная модель: DataFrame, разделы, задачи,
стадии, shuffle, ленивые планы и оптимизатор Catalyst — код переносится на кластер заменой `master` и путей к данным.

**2. Роли Driver и Executor.** Driver создаёт SparkSession, строит логический/физический план, разбивает его на стадии
и задачи, планирует их и собирает результаты actions. Executor выполняет задачи над разделами, хранит кэш и
промежуточные данные shuffle и возвращает результаты драйверу.

**3. Что такое partition и почему важно их число?** Partition — часть набора данных, обрабатываемая одной задачей.
Число разделов задаёт степень параллелизма: слишком мало — простаивают ядра и растёт объём одной задачи (риск нехватки
памяти), слишком много — накладные расходы на планирование и множество мелких файлов. В работе
`spark.sql.shuffle.partitions = 8` вместо 200 по умолчанию.

**4. Transformation vs action.** Transformation описывает новый DataFrame и не запускает вычисления:
`select()`, `filter()`, `withColumn()` (также `groupBy().agg()`, `join()`). Action требует результат и запускает задание:
`count()`, `show()`, `collect()` (также `write`, `first()`).

**5. Lazy evaluation.** Spark откладывает выполнение цепочки преобразований до action, чтобы оптимизировать план целиком
(проталкивание фильтров, отсечение столбцов, объединение операций). Поэтому `withColumn()` выполняется за миллисекунды —
он лишь добавляет узел в план; стоимость проявляется при `count()`/`show()`/`write` (см. демонстрацию в п. 5.3 части I).

**6. Почему CSV не содержит строгой схемы и когда нужен StructType?** CSV — просто текст с разделителями: типы, null-маркеры
и формат дат в нём не записаны. `inferSchema` требует дополнительного прохода по данным и может ошибиться (как с десятичной
запятой в наборе КОМУС). Явный `StructType` предпочтителен в повторяемых конвейерах, на больших файлах и когда тип нужно
зафиксировать как контракт (идентификаторы-строки с ведущими нулями, числа, даты).

**7. Зачем приводить ISO-дату к timestamp?** Лексикографическое сравнение работает только при строго одинаковом формате;
при этом невозможны `datediff()`, `year()`, `month()`, арифметика с интервалами, часовые пояса, и некорректные значения
не выявляются. Тип `timestamp/date` даёт корректную семантику и проверку формата (некорректная строка станет null).

**8. Почему null не означает удаление строки?** Пропуск часто отражает состояние процесса. 2 965 пропусков
`order_delivered_customer_date` почти целиком относятся к заказам со статусами shipped, canceled, unavailable и др. —
у них доставки ещё не было; удаление исказило бы статистику статусов. Аналогично в КОМУС пустой адрес означает продажу без доставки.

**9. `select()` vs `filter()`, `withColumn()` vs выбор столбца.** `select()` меняет набор столбцов (проекция), `filter()` —
набор строк (выборка). `withColumn()` добавляет или заменяет столбец вычисленным выражением, сохраняя все остальные;
простой выбор столбца лишь ссылается на существующие данные.

**10. Зачем сравнивать число строк до и после JOIN?** Рост числа строк указывает на неуникальный ключ (связь
многие-ко-многим) — как со справочником клиентов КОМУС; уменьшение при INNER JOIN — на потерю фактов из-за неполного
справочника. Без контроля агрегаты (выручка, количество) будут завышены или занижены.

**11. INNER JOIN vs LEFT JOIN для витрины фактов.** INNER оставляет только факты с парой в справочнике и молча теряет
остальные; LEFT сохраняет все факты, а отсутствие пары видно как null/`unknown`, что позволяет измерить неполноту справочника.
Для витрины выручки безопаснее LEFT.

**12. `COUNT(*)` vs `COUNT(DISTINCT order_id)`.** В `order_items` строка — позиция заказа, поэтому `COUNT(*)` — число
проданных позиций, а `COUNT(DISTINCT order_id)` — число заказов, содержащих категорию. Заказ с тремя позициями даёт 3 и 1.

**13. Spark SQL и DataFrame API.** Это два интерфейса к одному движку Spark SQL: оба строят логический план, который
оптимизирует Catalyst и исполняет один и тот же физический движок. Поэтому результаты совпадают (проверено `exceptAll`).

**14. Почему запись идёт в каталог с part-файлами?** Каждый раздел записывается своей задачей параллельно и независимо,
без сбора данных на одном узле; служебные файлы (`_SUCCESS`, `.crc`) подтверждают целостность записи. Каталог —
единица набора данных для последующего чтения.

**15. Почему `coalesce(1)` допустим только для маленьких данных?** Он сводит все данные в один раздел, и запись выполняет
одна задача на одном исполнителе. Для 74-строчной витрины это незаметно, а для больших данных — узкое место по времени,
памяти и диску, полностью теряющее параллелизм.

**16. Преимущества Parquet.** Колоночное хранение и сжатие (меньше объём), сохранённая схема и типы (не нужен
inferSchema), чтение только нужных столбцов (column pruning), пропуск блоков по статистикам min/max (predicate pushdown),
эффективное разбиение на части для параллельного чтения.